# Food Access Prioritization Pipeline — USDA 2019 LRAM
**Public Service & Food Access**

This notebook loads the USDA 2019 Large Retailer Access Map (LRAM), filters to a chosen county, cleans it, creates a prioritization score, and exports analysis-ready outputs.

**Design goal:** the county is a single config variable, so the whole analysis can be re-run for any other county without editing the logic.

> **Data note:** LRAM is historical — 2010 census-tract boundaries & population, 2014–18 ACS socioeconomics, and 2019 retailer locations. It is *not* a 2026 measure of food access.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# CHANGE THESE TWO LINES TO RE-RUN FOR ANOTHER AREA
STATE_NAME = "Illinois"
COUNTY_NAME = "Cook County"

# Validate state and county
def _require(name, value):
    # Raise a clear error if a required config value is missing or blank
    if value is None or str(value).strip() == "":
        raise ValueError(
            f"State and county must not be empty!")
    return str(value).strip()

# Call validation function
STATE_NAME = _require("STATE_NAME", STATE_NAME)
COUNTY_NAME = _require("COUNTY_NAME", COUNTY_NAME)

# For loading data
DATA_PATH = Path("FoodAccessResearchAtlasData2019.xlsx")
DATA_SHEET = "Food Access Research Atlas"

# For exporting data
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

# Variables used for the prioritization index
# A higher number means a higher need for all of these except MedianFamilyIncome
INDEX_VARS = [
    "PovertyRate",       # % population below federal poverty line
    "lapop1share",       # % of tract >1 mi from a supermarket
    "lalowi1share",      # % low-income AND >1 mi from a supermarket
    "lahunv1share",      # % housing units with no vehicle AND >1 mi
    "lasnap1share",      # % SNAP households AND >1 mi
    "MedianFamilyIncome" # Will invert: lower income = higher need
]
# Variables where a LOW value means MORE need and should have higher score
INVERT = ["MedianFamilyIncome"]

# Columns the USDA stores as the literal text "NULL" instead of a blank cell
NULLS = ["NULL", "null", "NA", "N/A", ""]

## 1. Load & filter

Load the dataset, read only the data sheet, then filter to the state and county. Keeping the filter inside the pipeline so the file is reusable.

In [3]:
# Function to load dataset and set NULL values to NaN
def load_lram(path, sheet=DATA_SHEET):
    #Read the LRAM data sheet, treating USDA 'NULL' text as missing
    return pd.read_excel(path, sheet_name=sheet, na_values=NULLS)

# Function to filter to State and county
def filter_area(df, state, county):
    out = df[df["State"] == state].copy()
    out = out[out["County"] == county].copy()
    return out.reset_index(drop=True)

# Check loaded dataset
df_all = load_lram(DATA_PATH)
df = filter_area(df_all, STATE_NAME, COUNTY_NAME)
print(f"National rows loaded: {len(df_all):,}")
print(f"Rows after filter: {len(df):,} ({COUNTY_NAME or 'all'}, {STATE_NAME})")
print(f"Columns: {df.shape[1]}")
df.head()

National rows loaded: 72,531
Rows after filter: 1,314 (Cook County, Illinois)
Columns: 147


,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,17031010100,Illinois,Cook County,1,4854,2302,0,218.0,4.491141,0,...,277.0,1810.0,2437.0,150.0,1.0,33.0,423.0,616.0,1162.0,433.0
1,17031010201,Illinois,Cook County,1,6450,2463,0,163.0,2.527132,0,...,314.0,2312.0,2350.0,299.0,6.0,48.0,1435.0,2049.0,751.0,851.0
2,17031010202,Illinois,Cook County,1,2818,1115,0,315.0,11.178141,0,...,309.0,1237.0,953.0,143.0,1.0,18.0,466.0,789.0,464.0,232.0
3,17031010300,Illinois,Cook County,1,6236,2826,0,791.0,12.684413,0,...,884.0,3267.0,1722.0,346.0,16.0,21.0,864.0,1169.0,993.0,470.0
4,17031010400,Illinois,Cook County,1,5042,2098,0,1349.0,26.755256,0,...,263.0,3341.0,729.0,562.0,5.0,17.0,388.0,480.0,655.0,277.0


For clarity, we will be refering to the following variables as the "share variables": lapop1share, lalowi1share, lahunv1share and lasnap1share.

## 2. Clean & type-check

The `na_values` were already converted `"NULL"` → `NaN` on load, so the share variables come in as numeric. This cell verifies that so nothing breaks later.


Census tract is set as a string because it is meant to be a label so we want to make sure all digits are kept and that it is 11 characters long.

In [4]:
# Force the analysis variables to numeric
# coerce to change any number that can't be turned to NaN (just in case)
for col in INDEX_VARS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep the tract id as a zero-padded string (it is a geographic code, not a number).
# Set type to string, remove decimal and keeps 11 digits (the size of a tract id)
df["CensusTract"] = df["CensusTract"].astype(str).str.split(".").str[0].str.zfill(11)

# Check data quality
quality = pd.DataFrame({
    "dtype": [df[c].dtype for c in INDEX_VARS],
    "missing": [df[c].isna().sum() for c in INDEX_VARS],
    "missing_%": [round(df[c].isna().mean()*100, 1) for c in INDEX_VARS],
    "min": [df[c].min() for c in INDEX_VARS],
    "max": [df[c].max() for c in INDEX_VARS],})

quality.index = INDEX_VARS
print("Data-quality summary for index variables:")
quality

Data-quality summary for index variables:


,dtype,missing,missing_%,min,max
PovertyRate,float64,0,0.0,0.293341,77.099587
lapop1share,float64,972,74.0,0.000000,100.000001
lalowi1share,float64,972,74.0,0.000000,87.673914
lahunv1share,float64,972,74.0,0.000000,53.865745
lasnap1share,float64,972,74.0,0.000000,62.007170
MedianFamilyIncome,float64,9,0.7,10131.000000,250001.000000


In the share variables a number is populated "only for tracts flagged low-access at 1 mile" so if the tract is not low-access at 1 mile it gets `NaN` meaning it doesn't apply so there is no number.

For scoring, `NaN` in the share variables will be treated as 0 to say "this tract has no low-access-at-1-mile population," which is what the blank actually means.

In [5]:
# Distance-share NaNs mean "not low-access" so they are 0 for scoring
# Create list for lapop1share, lalowi1share, lahunv1share, lasnap1share
DISTANCE_SHARES = [c for c in INDEX_VARS if c.startswith("la")]
# Replace NaN in those columns with 0
df[DISTANCE_SHARES] = df[DISTANCE_SHARES].fillna(0.0)

# MedianFamilyIncome/PovertyRate NaNs are unknowns, these houses have income
# Impute with county median so missing tracts don't drop out of the ranking
for col in ["MedianFamilyIncome", "PovertyRate"]:
    if col in df.columns:
        # Flag tracts a median was entered for
        df[col + "_imputed"] = df[col].isna()
        # Median for the selected county
        df[col] = df[col].fillna(df[col].median())

print("Remaining missing values in index vars:",
      int(df[INDEX_VARS].isna().sum().sum()))

Remaining missing values in index vars: 0


## 3. Build the prioritization score

**Approach: min–max normalize each variable to 0–1, then average.**

1. Put every variable on the same 0–1 scale (`(x − min) / (max − min)`).  
2. Invert income (low income = high need).
3. Average the normalized variables into one `PriorityScore` (0–1), with optional weights.


The `minmax` function rescales columns onto a 0-to-1 scale. It finds the column's minimum and maximum, then for each value computes (value − min) / (max − min). The lowest value in the column becomes 0, the highest becomes 1, and everything else lands in between.

In [6]:
# Function rescales columns onto a 0-to-1 scale
def minmax(s):
    lo, hi = s.min(), s.max()
    # If a column is all the same value, return all 0s
    if hi > lo:
        return (s - lo) / (hi - lo)
    else:
        return pd.Series(0.0, index=s.index)
        
# Equal weights by default but can be changed
WEIGHTS = {v: 1.0 for v in INDEX_VARS}

# Loop for each variable unsing minmax function
norm = pd.DataFrame(index=df.index)
for v in INDEX_VARS:
    n = minmax(df[v])
    # Invert for the MedianFamilyIncome so higher number gets lower scoring
    if v in INVERT:
        n = 1 - n
    norm[v + "_score"] = n

# In case variables are weighted differently
w = pd.Series(WEIGHTS)

# Calculate Priority Score and add column
df["PriorityScore"] = (norm.values * w.values).sum(axis=1) / w.sum()

# Rank each tract and add column
df["PriorityRank"]  = df["PriorityScore"].rank(ascending=False, method="min").astype(int)

# Adds the norm columns to the dataframe
df = pd.concat([df, norm], axis=1)

# Shows range from min to max
print("PriorityScore range:", round(df.PriorityScore.min(),3), "->", round(df.PriorityScore.max(),3))

# Check top ten tracts
df[["CensusTract","PriorityScore","PriorityRank"] + INDEX_VARS].sort_values("PriorityRank").head(10)

PriorityScore range: 0.003 -> 0.967


,CensusTract,PriorityScore,PriorityRank,PovertyRate,lapop1share,lalowi1share,lahunv1share,lasnap1share,MedianFamilyIncome
514,17031540102,0.967044,1,64.150943,100.000001,87.673914,53.865745,62.007170,17122.0
513,17031540101,0.884952,2,52.922318,92.261515,76.848514,51.205919,56.760179,19881.0
1269,17031838800,0.716158,3,42.514970,95.961718,57.497608,28.363315,41.911970,27083.0
1189,17031829700,0.624664,4,32.315764,96.304441,53.508441,17.252142,36.962597,48194.0
1042,17031821500,0.587037,5,40.681004,77.415171,49.390325,18.655457,29.057288,47563.0
1155,17031827700,0.579798,6,29.387187,100.000000,52.906824,8.917476,28.649237,41563.0
1026,17031820603,0.492289,7,20.894772,100.000001,38.236609,3.853674,25.352113,65545.0
1148,17031827000,0.484635,8,31.325625,67.461090,41.802874,10.868663,17.230334,40662.0
448,17031430700,0.483328,9,44.977376,47.513138,30.331301,25.749294,16.513481,69436.0
1154,17031827600,0.471462,10,25.409104,68.645694,36.044000,15.679582,17.724234,51563.0


## 4. View & Export

Top-priority tracts, a summary table, and analysis-ready files. The full ranked table is what feeds the report figures.

In [7]:
ranked = df.sort_values("PriorityRank")

# Define the columns we actually want in the outputs
ID_COLS = ["CensusTract", "State", "County", "Pop2010"]
SCORE_COLS = ["PriorityRank", "PriorityScore"]
NORM_COLS = [v + "_score" for v in INDEX_VARS]
FLAG_COLS = [c for c in ["PovertyRate_imputed", "MedianFamilyIncome_imputed"] if c in df.columns]

OUTPUT_COLS = ID_COLS + SCORE_COLS + INDEX_VARS + NORM_COLS + FLAG_COLS

# Keep only columns that actually exist (guards against a swapped county/variable set)
OUTPUT_COLS = [c for c in OUTPUT_COLS if c in ranked.columns]

ranked_out = ranked[OUTPUT_COLS]
top20_out = ranked_out.head(20)

print(f"Tracts scored: {len(df):,}")
print(f"Exporting {len(OUTPUT_COLS)} columns")

print("\nTop 10 priority tracts:")
# Show identifiers + score + ALL of the index variables (all four shares included)
display(top20_out[["CensusTract", "PriorityRank", "PriorityScore"] + INDEX_VARS].head(10))

slug = (COUNTY_NAME or STATE_NAME).lower().replace(" ", "_")
ranked_out.to_csv(OUT_DIR / f"{slug}_scored.csv", index=False)
top20_out.to_csv(OUT_DIR / f"{slug}_top20.csv", index=False)
print(f"\nWrote: {OUT_DIR / f'{slug}_scored.csv'} and _top20.csv")

Tracts scored: 1,314
Exporting 20 columns

Top 10 priority tracts:


,CensusTract,PriorityRank,PriorityScore,PovertyRate,lapop1share,lalowi1share,lahunv1share,lasnap1share,MedianFamilyIncome
514,17031540102,1,0.967044,64.150943,100.000001,87.673914,53.865745,62.007170,17122.0
513,17031540101,2,0.884952,52.922318,92.261515,76.848514,51.205919,56.760179,19881.0
1269,17031838800,3,0.716158,42.514970,95.961718,57.497608,28.363315,41.911970,27083.0
1189,17031829700,4,0.624664,32.315764,96.304441,53.508441,17.252142,36.962597,48194.0
1042,17031821500,5,0.587037,40.681004,77.415171,49.390325,18.655457,29.057288,47563.0
1155,17031827700,6,0.579798,29.387187,100.000000,52.906824,8.917476,28.649237,41563.0
1026,17031820603,7,0.492289,20.894772,100.000001,38.236609,3.853674,25.352113,65545.0
1148,17031827000,8,0.484635,31.325625,67.461090,41.802874,10.868663,17.230334,40662.0
448,17031430700,9,0.483328,44.977376,47.513138,30.331301,25.749294,16.513481,69436.0
1154,17031827600,10,0.471462,25.409104,68.645694,36.044000,15.679582,17.724234,51563.0



Wrote: outputs\cook_county_scored.csv and _top20.csv
